# Getting started with the NS Agent (ADK implementation)

This notebook walks through running the Neuro-Symbolic agent on a single FHIR bundle: build the chronological hypergraph, precompute node embeddings, instantiate the agent, and ask questions over the record.

## Setup

From the repository root, install the package and set a GenAI API key:

```bash
pip install -e .
export GENAI_API_KEY="your_gemini_api_key"
```

In [ ]:
import os
import json
import time
import datetime

from ns_agent_adk.core import graph as graph_module
from ns_agent_adk.core import linearizer as linearizer_module
from ns_agent_adk.config import config as config_module
from ns_agent_adk import engine as engine_module

In [ ]:
with open("sample_patient.json") as f:
    fhir_bundle = json.load(f)

print(f"Loaded bundle with {len(fhir_bundle['entry'])} resources.")

## Configure the models

The public GenAI API is used by default; set `GENAI_API_KEY` in the environment.

In [ ]:
API_KEY = os.getenv("GENAI_API_KEY")
if not API_KEY:
    raise ValueError("Set GENAI_API_KEY in your environment.")

config = config_module.Config(
    embedding_api_key=API_KEY,
    llm_api_key=API_KEY,
    embedding_model_name="gemini-embedding-001",
    llm_model_name="gemini-3-flash-preview",
    temporal_parser_model_name="gemini-3-flash-preview",
)

## Step 1: Build the hypergraph

In [ ]:
graph = graph_module.ChronologicalHypergraph()
graph.build_from_bundle(fhir_bundle)
print(f"Graph built with {len(graph.spine)} hypernodes in the spine.")

In [ ]:
# graph.spine[0] holds the first hypernode; graph.spine[0].nodes[0] its first node.

## Step 2: Precompute node embeddings

This runs once per graph.

In [ ]:
embedder = config.get_embedder()
precomputed_node_embeddings = linearizer_module.precompute_node_embeddings(graph, embedder)
print(f"Computed {len(precomputed_node_embeddings)} node embeddings.")

## Step 3: Instantiate the agent and ask questions

In [ ]:
ns_agent = engine_module.NeuroSymbolicAgent(
    config=config,
    graph=graph,
    precomputed_node_embeddings=precomputed_node_embeddings,
)

In [ ]:
query = "Does my latest cholesterol medication appear to be working?"
final_response, reasoning_trace = ns_agent.execute(query)
print(final_response)

In [ ]:
query = "Who prescribed my cholesterol medication and when?"
final_response, reasoning_trace = ns_agent.execute(query)
print(final_response)

In [ ]:
query = "What is the trend of my blood pressure?"
final_response, reasoning_trace = ns_agent.execute(query)
print(final_response)

## Inspect the reasoning trace

In [ ]:
for event in reasoning_trace:
    print(f"\n--- Turn (Author: {event.author}) ---")
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(f"Text Content:\n{part.text}")
    if event.actions:
        for action in event.actions:
            print(f"Triggered ADK Action / Tool Call: {action}")